# LLM Preference Prediction — Structural-Feature Baseline

## Problem

Given a conversation prompt and two LLM responses (`response_a`, `response_b`), predict which one a human evaluator would prefer — or whether they would call it a tie.  
This is a 3-class classification problem scored by **multiclass log loss** (lower is better).

The dataset comes from [LMSYS Chatbot Arena](https://lmsys.org/blog/2023-05-03-arena/) — real human preference votes between anonymised LLM pairs.  
Training set: **57,477 rows** across ~63 unique models.

## Key EDA Findings

Before building any model, a quick EDA revealed three structural signals worth knowing:

| Finding | Detail |
|---|---|
| **Class balance** | ~34.9% model_a wins / 34.2% model_b wins / 30.9% ties — roughly balanced |
| **Position bias** | model_a wins 0.72 pp more often than model_b — annotators show a slight primacy bias |
| **Verbosity bias** | The longer response wins ~43% vs ~26% for the shorter one — a **1.6× lift** for verbosity |

The verbosity bias is by far the strongest cheap feature: when `len_a > len_b`, model_a wins 43% of the time versus only 26% for model_b (a 16+ pp gap). This motivates a structural-feature baseline before investing in transformer inference.

## Feature Set and Approach

We extract **16 structural features** per row — no external models, no GPU, CPU-only in seconds:

| # | Feature | What it captures |
|---|---|---|
| 0–1 | `len_a`, `len_b` | Total character length of each response (after JSON parse) |
| 2 | `len_delta` | `len_a − len_b` — raw length advantage |
| 3 | `len_log_ratio` | `log((len_a+1)/(len_b+1))` — symmetric length ratio |
| 4–6 | `code_fences_a/b`, `code_fence_delta` | Markdown code block count — coding tasks may skew toward structured responses |
| 7–9 | `bullets_a/b`, `bullet_delta` | Bullet/numbered list lines — formatting richness |
| 10–12 | `headers_a/b`, `header_delta` | Markdown header lines — document structure |
| 13–14 | `overlap_a`, `overlap_b` | Token overlap of each response with the prompt (`|R ∩ P| / (|R|+1)`) |
| 15 | `turn_count` | Number of conversation turns (86.9% are single-turn) |

**Why a cheap interpretable floor first?**  
Before reaching for a fine-tuned LLM (slow, expensive, GPU-bound), it is worth knowing how far pure structural signals can get us. This baseline:
- Runs in under a minute on any CPU.
- Produces interpretable coefficients (Logistic Regression).
- Sets a concrete log-loss floor against which all future models are measured.
- Reveals which structural signals carry the most weight.

**Model:** `sklearn` Pipeline with `StandardScaler` → `LogisticRegression` (lbfgs, multinomial, C=1.0).

In [ ]:
from __future__ import annotations

import json
import logging
import math
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# Paths — Kaggle kernel environment
# ---------------------------------------------------------------------------
# Discover the competition data mount robustly: print what's actually
# mounted, then locate train.csv wherever it lives under /kaggle/input.
_input_root = Path("/kaggle/input")
for _p in sorted(_input_root.rglob("*"))[:20]:
    print("mounted:", _p)
_train_candidates = sorted(_input_root.rglob("train.csv"))
if not _train_candidates:
    raise FileNotFoundError(f"no train.csv anywhere under {_input_root}")
DATA_DIR = _train_candidates[0].parent
print("DATA_DIR resolved to:", DATA_DIR)
OUTPUT_PATH = Path("/kaggle/working/submission.csv")

LABEL_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]
N_FOLDS = 5
RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("baseline")

print("Imports complete. DATA_DIR:", DATA_DIR)

In [ ]:
# ---------------------------------------------------------------------------
# Data helpers
# ---------------------------------------------------------------------------

def safe_json_loads(s: Any) -> list[str]:
    """Parse a JSON-encoded list of strings, with fallback for malformed input.

    Returns a list of strings (empty list for null/missing values).
    """
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return []
    try:
        val = json.loads(s)
        if isinstance(val, list):
            return [t if t is not None else "" for t in val]
        return [str(val)]
    except (json.JSONDecodeError, TypeError, ValueError):
        return [str(s)]


def load_csv(path: Path) -> pd.DataFrame:
    """Load a CSV, raising with context if it's missing."""
    if not path.exists():
        log.error("File not found: %s", path)
        raise FileNotFoundError(f"missing: {path}")
    log.info("Loading %s ...", path)
    return pd.read_csv(path)


# Load data
train_df = load_csv(DATA_DIR / "train.csv")
test_df = load_csv(DATA_DIR / "test.csv")
sample_sub = load_csv(DATA_DIR / "sample_submission.csv")

log.info("train rows=%d  test rows=%d", len(train_df), len(test_df))
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
train_df.head(3)

In [ ]:
# ---------------------------------------------------------------------------
# Labels
# ---------------------------------------------------------------------------

# LABEL_COLS order: [winner_model_a, winner_model_b, winner_tie] → classes 0, 1, 2
y = np.argmax(train_df[LABEL_COLS].values, axis=1)

class_counts = dict(zip(LABEL_COLS, np.bincount(y)))
log.info("Class distribution: %s", class_counts)
print("\nClass distribution:")
for label, count in class_counts.items():
    print(f"  {label:20s}: {count:6d}  ({count/len(y)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------

def _count_code_fences(text: str) -> int:
    """Count occurrences of markdown code fences (``` or ~~~)."""
    return len(re.findall(r"^```|^~~~", text, re.MULTILINE))


def _count_bullet_lines(text: str) -> int:
    """Count lines starting with a bullet marker (-, *, +, or numbered lists)."""
    return len(re.findall(r"^\s*[-*+]\s|^\s*\d+\.\s", text, re.MULTILINE))


def _count_header_lines(text: str) -> int:
    """Count markdown header lines (# ... through ######)."""
    return len(re.findall(r"^#{1,6}\s", text, re.MULTILINE))


def _token_overlap(response_text: str, prompt_text: str) -> float:
    """Cheap word-token set overlap: |R ∩ P| / (|R| + 1).

    Measures how many unique words in the response also appear in the prompt,
    normalised by response vocabulary size.
    """
    r_tokens = set(re.findall(r"\w+", response_text.lower()))
    p_tokens = set(re.findall(r"\w+", prompt_text.lower()))
    if not r_tokens:
        return 0.0
    return len(r_tokens & p_tokens) / (len(r_tokens) + 1)


def extract_features(df: pd.DataFrame) -> np.ndarray:
    """Build feature matrix from a DataFrame containing prompt/response columns.

    Returns an ndarray of shape (n_rows, 16). All features are numeric
    and will be standardised downstream via the Pipeline.

    Feature layout (16 features):
        0  len_a          - total char length of response_a (all turns)
        1  len_b          - total char length of response_b (all turns)
        2  len_delta      - len_a - len_b
        3  len_log_ratio  - log((len_a+1)/(len_b+1))
        4  code_fences_a  - code fence count in response_a
        5  code_fences_b  - code fence count in response_b
        6  code_fence_delta - code_fences_a - code_fences_b
        7  bullets_a      - bullet lines in response_a
        8  bullets_b      - bullet lines in response_b
        9  bullet_delta   - bullets_a - bullets_b
        10 headers_a      - header lines in response_a
        11 headers_b      - header lines in response_b
        12 header_delta   - headers_a - headers_b
        13 overlap_a      - token overlap of response_a with prompt
        14 overlap_b      - token overlap of response_b with prompt
        15 turn_count     - number of prompt turns
    """
    rows: list[list[float]] = []

    log.info("Extracting features from %d rows ...", len(df))

    for _, row in df.iterrows():
        prompts = safe_json_loads(row.get("prompt", ""))
        resp_a = safe_json_loads(row.get("response_a", ""))
        resp_b = safe_json_loads(row.get("response_b", ""))

        text_a = " ".join(resp_a)
        text_b = " ".join(resp_b)
        text_p = " ".join(prompts)

        len_a = len(text_a)
        len_b = len(text_b)
        len_delta = len_a - len_b
        len_log_ratio = math.log((len_a + 1) / (len_b + 1))

        code_a = _count_code_fences(text_a)
        code_b = _count_code_fences(text_b)

        bullets_a = _count_bullet_lines(text_a)
        bullets_b = _count_bullet_lines(text_b)

        headers_a = _count_header_lines(text_a)
        headers_b = _count_header_lines(text_b)

        overlap_a = _token_overlap(text_a, text_p)
        overlap_b = _token_overlap(text_b, text_p)

        turn_count = len(prompts)

        rows.append([
            len_a, len_b, len_delta, len_log_ratio,
            code_a, code_b, code_a - code_b,
            bullets_a, bullets_b, bullets_a - bullets_b,
            headers_a, headers_b, headers_a - headers_b,
            overlap_a, overlap_b,
            turn_count,
        ])

    return np.array(rows, dtype=np.float64)


print("Extracting train features...")
X_train = extract_features(train_df)
print(f"Train feature matrix: {X_train.shape}")

print("\nExtracting test features...")
X_test = extract_features(test_df)
print(f"Test feature matrix:  {X_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Model building and evaluation helpers
# ---------------------------------------------------------------------------

def build_pipeline() -> Pipeline:
    """Construct the sklearn Pipeline: StandardScaler -> LogisticRegression."""
    # Note: multi_class="multinomial" was removed in scikit-learn 1.5+.
    # With solver="lbfgs" and >2 classes the multinomial loss is used automatically.
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            C=1.0,
            random_state=RANDOM_STATE,
        )),
    ])


def class_prior_log_loss(y: np.ndarray) -> float:
    """Log loss when always predicting the training class prior probabilities.

    This is the floor against which the model is compared — the trivial
    baseline of always predicting the observed class frequencies.
    """
    n = len(y)
    classes, counts = np.unique(y, return_counts=True)
    priors = counts / n
    # Broadcast: every row gets the same prior vector
    y_pred = np.tile(priors, (n, 1))
    return float(log_loss(y, y_pred))


def cross_validate(
    X: np.ndarray,
    y: np.ndarray,
    n_folds: int = N_FOLDS,
) -> tuple[float, float]:
    """Run stratified k-fold CV; return (mean_log_loss, std_log_loss)."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_losses: list[float] = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        pipe = build_pipeline()
        pipe.fit(X_tr, y_tr)
        proba = pipe.predict_proba(X_val)
        loss = log_loss(y_val, proba)
        fold_losses.append(loss)
        log.info("  Fold %d/%d  log_loss=%.5f", fold_idx, n_folds, loss)

    mean_ll = float(np.mean(fold_losses))
    std_ll = float(np.std(fold_losses))
    return mean_ll, std_ll


# Class-prior floor
prior_ll = class_prior_log_loss(y)
log.info("Class-prior log loss (floor): %.5f", prior_ll)

# 5-fold stratified CV
log.info("Running %d-fold stratified CV ...", N_FOLDS)
mean_ll, std_ll = cross_validate(X_train, y)

print("\n" + "=" * 60)
print("CV RESULTS")
print("=" * 60)
print(f"  Class-prior log loss (floor) : {prior_ll:.5f}")
print(f"  CV mean log loss (5-fold)    : {mean_ll:.5f}")
print(f"  CV std                       : {std_ll:.5f}")
print(f"  Improvement over prior       : {prior_ll - mean_ll:.5f}")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------------------------
# Fit final model on full training set and write submission
# ---------------------------------------------------------------------------

log.info("Fitting final model on full training set ...")
final_model = build_pipeline()
final_model.fit(X_train, y)

# Predict on test set
proba = final_model.predict_proba(X_test)
classes = final_model.classes_  # [0, 1, 2] mapped to LABEL_COLS order

# Build prediction DataFrame aligned to LABEL_COLS
pred_df = pd.DataFrame({
    "id": test_df["id"].values,
    LABEL_COLS[int(classes[0])]: proba[:, 0],
    LABEL_COLS[int(classes[1])]: proba[:, 1],
    LABEL_COLS[int(classes[2])]: proba[:, 2],
})

# Re-order columns and rows to match sample submission
expected_cols = list(sample_sub.columns)
pred_df = pred_df[expected_cols]

id_order = sample_sub["id"].tolist()
pred_df = pred_df.set_index("id").reindex(id_order).reset_index()

# Write submission
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(OUTPUT_PATH, index=False)
log.info("Submission written to %s (%d rows)", OUTPUT_PATH, len(pred_df))

print(f"\nSubmission saved to: {OUTPUT_PATH}")
print(f"Shape: {pred_df.shape}")
print("\nFirst 5 rows:")
pred_df.head()